In [1]:
# =====================================
# ASL CNN MODEL TRAINING
# =====================================

import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten,
    Dense, Dropout, Input
)
from sklearn.model_selection import train_test_split
from collections import Counter

# -------- PATHS --------
BASE_DIR = r"C:\Users\Dell\Desktop\asl-sign-recognition"
MODEL_DIR = os.path.join(BASE_DIR, "models")
# -----------------------

# Load preprocessed data
X = np.load(os.path.join(MODEL_DIR, "X.npy"))
y = np.load(os.path.join(MODEL_DIR, "y.npy"))
labels = np.load(os.path.join(MODEL_DIR, "labels.npy"), allow_pickle=True)

print("Data loaded:")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", labels)

# -----------------------
# Check class distribution
# -----------------------
print("Class distribution:", Counter(y))

# Train-test split (80-20 as per report)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# -----------------------
# Data Augmentation
# -----------------------
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

# -----------------------
# CNN Model
# -----------------------
model = Sequential([
    Input(shape=(64, 64, 1)),
    data_augmentation,

    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(len(labels), activation='softmax')
])

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# -----------------------
# Train Model
# -----------------------
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_data=(X_test, y_test),
    shuffle=True
)

# -----------------------
# Save Model
# -----------------------
os.makedirs(MODEL_DIR, exist_ok=True)

model.save(os.path.join(MODEL_DIR, "asl_model_full.keras"))

print("✅ Model training completed and saved")


Data loaded:
X shape: (31465, 64, 64, 1)
y shape: (31465,)
Classes: ['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'R' 'Sorry' 'hello' 'please']
Class distribution: Counter({np.int64(3): 2501, np.int64(6): 2500, np.int64(7): 2500, np.int64(8): 2500, np.int64(9): 2500, np.int64(5): 2499, np.int64(11): 2499, np.int64(12): 2499, np.int64(2): 2493, np.int64(0): 2485, np.int64(4): 2481, np.int64(10): 2057, np.int64(1): 1951})


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (None, 64, 64, 1)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d (Conv2D)                      │ (None, 62, 62, 32)          │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 31, 31, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 29, 29, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 14, 14, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 12, 12, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 6, 6, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 4608)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         589,952 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 13)                  │           1,677 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 684,301 (2.61 MB)

 Trainable params: 684,301 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
787/787 ━━━━━━━━━━━━━━━━━━━━ 144s 174ms/step - accuracy: 0.0770 - loss: 2.5745 - val_accuracy: 0.1179 - val_loss: 2.5598
Epoch 2/15
787/787 ━━━━━━━━━━━━━━━━━━━━ 152s 193ms/step - accuracy: 0.0809 - loss: 2.5637 - val_accuracy: 0.1017 - val_loss: 2.5599
Epoch 3/15
787/787 ━━━━━━━━━━━━━━━━━━━━ 192s 181ms/step - accuracy: 0.0795 - loss: 2.5634 - val_accuracy: 0.1613 - val_loss: 2.4970
Epoch 4/15
787/787 ━━━━━━━━━━━━━━━━━━━━ 218s 202ms/step - accuracy: 0.0809 - loss: 2.5627 - val_accuracy: 0.1116 - val_loss: 2.4851
Epoch 5/15
787/787 ━━━━━━━━━━━━━━━━━━━━ 106s 79ms/step - accuracy: 0.0814 - loss: 2.5620 - val_accuracy: 0.2245 - val_loss: 2.2848
Epoch 6/15
787/787 ━━━━━━━━━━━━━━━━━━━━ 62s 79ms/step - accuracy: 0.0842 - loss: 2.5606 - val_accuracy: 0.2295 - val_loss: 2.2682
Epoch 7/15
787/787 ━━━━━━━━━━━━━━━━━━━━ 82s 79ms/step - accuracy: 0.0820 - loss: 2.5588 - val_accuracy: 0.3221 - val_loss: 2.0952
Epoch 8/15
787/787 ━━━━━━━━━━━━━━━━━━━━ 65s 82ms/step - accuracy: 0.0899 - loss: 